# 📉 04. Statistical Baseline Forecasting: Naive & Facebook Prophet
**Capstone Project DS-02: Demand Forecasting & Inventory Optimization**

Evaluates classical baseline methods on the 28-day out-of-time test horizon:
- Seasonal Naive (7-day periodic repeat)
- Moving Average (trailing 7-day average)
- Facebook Prophet (additive trend + seasonality + exogenous price regressors)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data.data_loader import M5DataLoader
from src.features.build_features import FeaturePipeline
from src.models.baselines import SeasonalNaiveModel, MovingAverageModel
from src.models.prophet_model import ProphetForecaster
from src.models.metrics import evaluate_forecast

loader = M5DataLoader(data_dir='../data/sample')
df = loader.build_merged_dataset(max_days=150)
pipeline = FeaturePipeline(lags=[7, 14, 28], rolling_windows=[7, 14])
feat_df = pipeline.fit_transform(df)
train_df, val_df = loader.split_train_val(feat_df, horizon_days=28)


In [ ]:
# Fit and evaluate Seasonal Naive
sample_series = train_df.groupby('date')['demand'].sum().values
y_val = val_df.groupby('date')['demand'].sum().values[:28]

snaive = SeasonalNaiveModel(season_length=7).fit(sample_series)
snaive_preds = snaive.predict(horizon=28)['point']

snaive_metrics = evaluate_forecast(y_val, snaive_preds, y_train_history=sample_series)
print('Seasonal Naive Metrics:', snaive_metrics)
